# Topic modeling: LDA baseline and BERTopic

**Owner:** Ibraheem

Skeleton only: fill in the empty code cells; do not change contracts.

In [1]:
# Data paths: run this cell first.
import sys
from pathlib import Path

REPO_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
DATA_PATH = REPO_DIR / "data" / "papers.parquet"
EMBEDDINGS_PATH = REPO_DIR / "data" / "embeddings.npy"
DATA_PATH, EMBEDDINGS_PATH


(PosixPath('/media/ibraheem/HDD/research_paper_discovery/data/papers.parquet'),
 PosixPath('/media/ibraheem/HDD/research_paper_discovery/data/embeddings.npy'))

## Load cleaned abstracts

Load the lemmatized text from data/papers.parquet.

In [2]:
import numpy as np
import pandas as pd

papers = pd.read_parquet(DATA_PATH, columns=["id", "title", "abstract", "text_lemma"])
texts_lemma = papers["text_lemma"].astype(str).tolist()
tokenized_texts = [t.split() for t in texts_lemma]
print(f"papers: {len(papers):,}")
print(f"columns: {list(papers.columns)}")
print(f"first title: {papers['title'].iloc[0][:100]}")


papers: 119,919
columns: ['id', 'title', 'abstract', 'text_lemma']
first title: Error-Robust Multi-View Clustering


## Tune LDA topic count

Compare c_v coherence for several k values.

In [3]:
# k=20 is fixed by the experiment specification. No k sweep is run: a
# full-corpus sweep would multiply the training cost, and a sampled sweep
# would silently subsample. c_v is reported for this model below.
K = 20
print(f"Using spec-fixed k={K}; c_v is computed after fitting on all papers.")


Using spec-fixed k=20; c_v is computed after fitting on all papers.


## Fit the final LDA model

Train the selected k-topic model on all papers.

In [4]:
import time
from src.topics import run_lda

t0 = time.perf_counter()
lda = run_lda(texts_lemma, k=K, seed=42)
print(f"LDA runtime: {time.perf_counter() - t0:.1f}s")
print(f"topics: {lda['k']}, model: {lda['model_path']}")
for t, words in enumerate(lda["topic_words"]):
    print(f"LDA topic {t}: {', '.join(words)}")


LDA runtime: 201.0s
topics: 20, model: /media/ibraheem/HDD/research_paper_discovery/cache/lda_model
LDA topic 0: agent, learning, bias, reinforcement, environment, learn, social, policy, human, fairness
LDA topic 1: image, pose, reconstruction, view, shape, estimation, quality, human, feature, high
LDA topic 2: detection, object, segmentation, point, feature, cloud, image, semantic, anomaly, detector
LDA topic 3: network, neural, layer, architecture, feature, attention, memory, accuracy, efficient, deep
LDA topic 4: temporal, scene, motion, event, depth, view, spatial, camera, dynamic, multi
LDA topic 5: graph, network, neural, learning, node, learn, structure, deep, representation, datum
LDA topic 6: adversarial, attack, privacy, robustness, training, learning, datum, client, federated, perturbation
LDA topic 7: domain, knowledge, learning, transfer, target, adaptation, feature, distillation, learn, source
LDA topic 8: video, recognition, face, action, image, facial, gan, frame, gener

## Fit BERTopic

Use the Kaggle embeddings with BERTopic and reduce topics.

In [5]:
from src.topics import run_bertopic

embeddings = np.load(EMBEDDINGS_PATH, mmap_mode="r")
print(f"embeddings shape: {embeddings.shape}")
t0 = time.perf_counter()
bt = run_bertopic(texts_lemma, embeddings, nr_topics=25)
print(f"BERTopic runtime: {time.perf_counter() - t0:.1f}s")
print(f"topics: {bt['nr_topics']}, outlier share: {bt['outlier_share']:.4f}")
for tid, words in zip(bt["topic_ids"], bt["topic_words"]):
    print(f"BERTopic {tid}: {', '.join(words)}")


embeddings shape: (119919, 384)


2026-11-22 15:07:55,674 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm


2026-11-22 15:11:54,936 - BERTopic - Dimensionality - Completed ✓


2026-11-22 15:11:55,054 - BERTopic - Cluster - Start clustering the reduced embeddings


2026-11-22 15:12:19,678 - BERTopic - Cluster - Completed ✓


2026-11-22 15:12:19,682 - BERTopic - Representation - Extracting topics using c-TF-IDF for topic reduction.


... (3 outputs left out)

2026-11-22 15:12:42,918 - BERTopic - Representation - Completed ✓


2026-11-22 15:12:42,940 - BERTopic - Topic reduction - Reduced number of topics from 258 to 25


2026-11-22 15:12:44,567 - BERTopic - WARNING: When you use `pickle` to save/load a BERTopic model,please make sure that the environments in which you saveand load the model are **exactly** the same. The version of BERTopic,its dependencies, and python need to remain the same.


BERTopic runtime: 509.3s
topics: 24, outlier share: 0.4282
BERTopic 0: language, model, text, translation, large, dataset, base, question, word, system
BERTopic 1: image, segmentation, detection, label, anomaly, feature, learning, domain, object, dataset
BERTopic 2: network, neural, learning, architecture, training, gradient, layer, model, quantization, weight
BERTopic 3: emotion, sentiment, bias, language, model, social, analysis, gender, dataset, legal
BERTopic 4: policy, reinforcement, learning, agent, reward, algorithm, environment, learn, action, offline
BERTopic 5: image, diffusion, gan, model, generation, generative, video, quality, generate, motion
BERTopic 6: adversarial, attack, robustness, model, client, perturbation, learning, federated, training, privacy
BERTopic 7: pose, point, cloud, estimation, object, depth, view, image, human, feature
BERTopic 8: series, time, forecasting, model, neural, datum, prediction, network, learning, pde
BERTopic 9: bandit, regret, algorithm, 

## Label topics

Turn top words into short human-readable labels.

In [6]:
from IPython.display import Markdown, display
from src.topics import coherence_cv, label_topics

lda_c_v = coherence_cv(lda["topic_words"], tokenized_texts)
bt_c_v = coherence_cv(bt["topic_words"], tokenized_texts)
print(f"LDA c_v: {lda_c_v:.4f}")
print(f"BERTopic c_v: {bt_c_v:.4f}")

lda_labels = label_topics(lda["topic_words"])
bt_label_map = label_topics(bt["topic_words"])
bt_labels = {tid: bt_label_map[i] for i, tid in enumerate(bt["topic_ids"])}
bt_labels[-1] = "Outliers"

display(Markdown(
    "| Model | c_v | topics | outlier share |\n"
    "|---|---|---|---|\n"
    f"| LDA | {lda_c_v:.4f} | {lda['k']} | 0.0000 |\n"
    f"| BERTopic | {bt_c_v:.4f} | {bt['nr_topics']} | {bt['outlier_share']:.4f} |"
))

lda_md = ["| topic | label | top words |", "|---|---|---|"]
for t in range(lda["k"]):
    lda_md.append(f"| {t} | {lda_labels[t]} | {', '.join(lda['topic_words'][t][:5])} |")
display(Markdown("### LDA labels\n" + "\n".join(lda_md)))

bt_md = ["| topic | label | top words |", "|---|---|---|"]
for tid in bt["topic_ids"]:
    i = bt["topic_ids"].index(tid)
    bt_md.append(f"| {tid} | {bt_labels[tid]} | {', '.join(bt['topic_words'][i][:5])} |")
display(Markdown("### BERTopic labels\n" + "\n".join(bt_md)))

lda_arr = np.asarray(lda["doc_topics"])
bt_arr = np.asarray(bt["doc_topics"])
for name, arr, labels in (("LDA", lda_arr, lda_labels), ("BERTopic", bt_arr, bt_labels)):
    print(f"\n3 sample documents for 5 {name} topics:")
    for tid in list(labels)[:5]:
        idxs = np.where(arr == tid)[0][:3]
        print(f"  {name} topic {tid} ({labels[tid]}):")
        for j in idxs:
            print(f"    - {papers['title'].iloc[int(j)][:110]}")


LDA c_v: 0.5605
BERTopic c_v: 0.5629


| Model | c_v | topics | outlier share |
|---|---|---|---|
| LDA | 0.5605 | 20 | 0.0000 |
| BERTopic | 0.5629 | 24 | 0.4282 |

### LDA labels
| topic | label | top words |
|---|---|---|
| 0 | Agent / Learning / Bias | agent, learning, bias, reinforcement, environment |
| 1 | Image / Pose / Reconstruction | image, pose, reconstruction, view, shape |
| 2 | Detection / Object / Segmentation | detection, object, segmentation, point, feature |
| 3 | Network / Neural / Layer | network, neural, layer, architecture, feature |
| 4 | Temporal / Scene / Motion | temporal, scene, motion, event, depth |
| 5 | Graph / Network / Neural | graph, network, neural, learning, node |
| 6 | Adversarial / Attack / Privacy | adversarial, attack, privacy, robustness, training |
| 7 | Domain / Knowledge / Learning | domain, knowledge, learning, transfer, target |
| 8 | Video / Recognition / Face | video, recognition, face, action, image |
| 9 | Question / Answer / Reasoning | question, answer, reasoning, evaluation, human |
| 10 | User / Dialogue / System | user, dialogue, system, response, conversation |
| 11 | Datum / Training / Label | datum, training, label, learning, sample |
| 12 | Language / Translation / Text | language, translation, text, speech, resource |
| 13 | Image / Medical / Clinical | image, medical, clinical, learning, datum |
| 14 | Image / Text / Generation | image, text, generation, diffusion, semantic |
| 15 | Algorithm / Learning / Optimization | algorithm, learning, optimization, function, distribution |
| 16 | Language / Llms / Llm | language, llms, llm, large, prompt |
| 17 | Datum / Prediction / Learning | datum, prediction, learning, research, machine |
| 18 | Self / Learning / Supervise | self, learning, supervise, attention, label |
| 19 | Language / Word / Analysis | language, word, analysis, sentiment, bias |

### BERTopic labels
| topic | label | top words |
|---|---|---|
| 0 | Language / Model / Text | language, model, text, translation, large |
| 1 | Image / Segmentation / Detection | image, segmentation, detection, label, anomaly |
| 2 | Network / Neural / Learning | network, neural, learning, architecture, training |
| 3 | Emotion / Sentiment / Bias | emotion, sentiment, bias, language, model |
| 4 | Policy / Reinforcement / Learning | policy, reinforcement, learning, agent, reward |
| 5 | Image / Diffusion / Gan | image, diffusion, gan, model, generation |
| 6 | Adversarial / Attack / Robustness | adversarial, attack, robustness, model, client |
| 7 | Pose / Point / Cloud | pose, point, cloud, estimation, object |
| 8 | Series / Time / Forecasting | series, time, forecasting, model, neural |
| 9 | Bandit / Regret / Algorithm | bandit, regret, algorithm, optimization, arm |
| 10 | Graph / Node / Gnn | graph, node, gnn, neural, network |
| 11 | Action / Video / Recognition | action, video, recognition, sign, temporal |
| 12 | Fairness / Fair / Decision | fairness, fair, decision, class, bias |
| 13 | Traffic / Trajectory / Prediction | traffic, trajectory, prediction, temporal, pedestrian |
| 14 | Face / Recognition / Image | face, recognition, image, deepfake, gaze |
| 15 | Molecular / Molecule / Protein | molecular, molecule, protein, drug, chemical |
| 16 | Causal / Imputation / Treatment | causal, imputation, treatment, variable, effect |
| 17 | Clustering / Cluster / View | clustering, cluster, view, algorithm, datum |
| 18 | Hash / Food / Sketch | hash, food, sketch, image, retrieval |
| 19 | Tensor / Matrix / Factorization | tensor, matrix, factorization, rank, completion |
| 20 | Quantum / Classical / Machine | quantum, classical, machine, hybrid, circuit |
| 21 | Radar / Sar / Fusion | radar, sar, fusion, camera, sensor |
| 22 | Emergent / Agent / Communication | emergent, agent, communication, language, emergence |
| 23 | Patent / Claim / Text | patent, claim, text, model, technology |


3 sample documents for 5 LDA topics:
  LDA topic 0 (Agent / Learning / Bias):


    - Faster Deep Q-learning using Neural Episodic Control
    - Using reinforcement learning to learn how to play text-based games
    - Group Communication Analysis: A Computational Linguistics Approach for Detecting Sociocognitive Roles in Multi
  LDA topic 1 (Image / Pose / Reconstruction):
    - Deep Spatial Feature Reconstruction for Partial Person Re-identification: Alignment-Free Approach
    - Polynomial-based rotation invariant features
    - Brain MRI Super Resolution Using 3D Deep Densely Connected Neural Networks
  LDA topic 2 (Detection / Object / Segmentation):
    - Unsupervised Object-Level Video Summarization with Online Motion Auto-Encoder
    - FOTS: Fast Oriented Text Spotting with a Unified Network
    - End-to-end detection-segmentation network with ROI convolution
  LDA topic 3 (Network / Neural / Layer):
    - Script Identification in Natural Scene Image and Video Frame using Attention based Convolutional-LSTM Network
    - Depth-Adaptive Computational Policies

## Save assignments

Write data/doc_topics.parquet for the trend notebook.

In [7]:
import json
from src.settings_p2 import DOC_TOPICS_PATH, ROOT

assign = pd.DataFrame({
    "id": papers["id"].to_numpy(),
    "lda_topic": np.asarray(lda["doc_topics"], dtype=np.int64),
    "bertopic_topic": np.asarray(bt["doc_topics"], dtype=np.int64),
})
assert len(assign) == len(papers) and assign["id"].equals(papers["id"].reset_index(drop=True))
assign.to_parquet(DOC_TOPICS_PATH, index=False)
print(f"wrote {DOC_TOPICS_PATH} ({len(assign):,} rows)")

lda_json = {
    "model": "gensim LDA", "k": lda["k"], "seed": lda["seed"],
    "c_v": round(lda_c_v, 4), "outlier_share": 0.0,
    "topics": [
        {"topic_id": t, "label": lda_labels[t], "top_words": lda["topic_words"][t]}
        for t in range(lda["k"])
    ],
}
bt_json = {
    "model": "BERTopic", "nr_topics": bt["nr_topics"], "nr_topics_requested": 25,
    "c_v": round(bt_c_v, 4), "outlier_share": round(bt["outlier_share"], 4),
    "topics": [
        {"topic_id": tid, "label": bt_labels[tid], "top_words": bt["topic_words"][i]}
        for i, tid in enumerate(bt["topic_ids"])
    ],
}
for name, payload in (("topics_lda.json", lda_json), ("topics_bertopic.json", bt_json)):
    (ROOT / "results" / name).write_text(json.dumps(payload, indent=2), encoding="utf-8")
    print(f"wrote results/{name}")


wrote /media/ibraheem/HDD/research_paper_discovery/data/doc_topics.parquet (119,919 rows)
wrote results/topics_lda.json
wrote results/topics_bertopic.json


## Results

LDA produced **20** topics with c_v **0.5605**; BERTopic produced **24** non-outlier topics with c_v **0.5629** and an outlier share of **0.4282**. Both scores use the same c_v function on the same tokenized `text_lemma`, so the small gap reflects the representations rather than different inputs: LDA's bag-of-words topics are broad and overlap (for example, several topics share `network`, `neural` and `learning`), while BERTopic's embedding-based clusters separate finer themes such as language/models, image segmentation, and emotion/sentiment. The 42.82% outlier share means a large fraction of papers were left unassigned by HDBSCAN; those rows carry `-1` in `data/doc_topics.parquet` and must be excluded from topic-share calculations. Runtime on this machine (20 CPUs, no GPU): LDA 201.0s, BERTopic 509.3s. Labels are drafts from the top three words and are awaiting human review.
